# SC-5-Token-Standards - Standards de Tokens

**Navigation** : [Index](../README.md) | [<< Errors](../01-Solidity-Foundation/SC-4-Errors-Events.ipynb) | [DeFi >>](SC-6-DeFi-Primitives.ipynb)

---

## Objectifs d'apprentissage

1. Implementer le standard **ERC-20** (tokens fongibles)
2. Implementer le standard **ERC-721** (NFTs)
3. Decouvrir **ERC-1155** (multi-tokens)
4. Utiliser les contrats OpenZeppelin

### Duree estimee : 50 minutes

---

## 1. Standard ERC-20

ERC-20 est le standard pour les tokens fongibles (interchangeables).

In [ ]:
# Interface ERC-20 complete
ERC20_INTERFACE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

/// @title ERC-20 Token Standard
interface IERC20 {
    // Events
    event Transfer(address indexed from, address indexed to, uint256 value);
    event Approval(address indexed owner, address indexed spender, uint256 value);

    // View functions
    function totalSupply() external view returns (uint256);
    function balanceOf(address account) external view returns (uint256);
    function allowance(address owner, address spender) external view returns (uint256);

    // Transfer functions
    function transfer(address to, uint256 amount) external returns (bool);
    function transferFrom(address from, address to, uint256 amount) external returns (bool);

    // Approval function
    function approve(address spender, uint256 amount) external returns (bool);
}
'''

print("Interface ERC-20:")
print(ERC20_INTERFACE)

In [ ]:
# Implementation complete ERC-20
ERC20_IMPLEMENTATION = '''
contract SimpleERC20 is IERC20 {
    string public constant name = "Simple Token";
    string public constant symbol = "SIM";
    uint8 public constant decimals = 18;

    uint256 private _totalSupply;
    mapping(address => uint256) private _balances;
    mapping(address => mapping(address => uint256)) private _allowances;

    constructor(uint256 initialSupply) {
        _totalSupply = initialSupply * 10 ** uint256(decimals);
        _balances[msg.sender] = _totalSupply;
        emit Transfer(address(0), msg.sender, _totalSupply);
    }

    function totalSupply() external view override returns (uint256) {
        return _totalSupply;
    }

    function balanceOf(address account) external view override returns (uint256) {
        return _balances[account];
    }

    function allowance(address owner, address spender) 
        external view override returns (uint256) 
    {
        return _allowances[owner][spender];
    }

    function transfer(address to, uint256 amount) external override returns (bool) {
        _transfer(msg.sender, to, amount);
        return true;
    }

    function approve(address spender, uint256 amount) external override returns (bool) {
        _allowances[msg.sender][spender] = amount;
        emit Approval(msg.sender, spender, amount);
        return true;
    }

    function transferFrom(address from, address to, uint256 amount) 
        external override returns (bool) 
    {
        uint256 currentAllowance = _allowances[from][msg.sender];
        require(currentAllowance >= amount, "ERC20: insufficient allowance");
        _allowances[from][msg.sender] = currentAllowance - amount;
        _transfer(from, to, amount);
        return true;
    }

    function _transfer(address from, address to, uint256 amount) internal {
        require(from != address(0), "ERC20: transfer from zero address");
        require(to != address(0), "ERC20: transfer to zero address");
        require(_balances[from] >= amount, "ERC20: insufficient balance");
        
        _balances[from] -= amount;
        _balances[to] += amount;
        emit Transfer(from, to, amount);
    }

    // Mint (internal)
    function _mint(address to, uint256 amount) internal {
        require(to != address(0), "ERC20: mint to zero address");
        _totalSupply += amount;
        _balances[to] += amount;
        emit Transfer(address(0), to, amount);
    }

    // Burn (internal)
    function _burn(address from, uint256 amount) internal {
        require(from != address(0), "ERC20: burn from zero address");
        require(_balances[from] >= amount, "ERC20: burn amount exceeds balance");
        _balances[from] -= amount;
        _totalSupply -= amount;
        emit Transfer(from, address(0), amount);
    }
}
'''

print("Implementation ERC-20 complete:")
print(ERC20_IMPLEMENTATION)

---

## 2. Standard ERC-721 (NFT)

ERC-721 est le standard pour les tokens non-fongibles (uniques).

In [ ]:
# Interface ERC-721
ERC721_INTERFACE = '''
interface IERC721 {
    event Transfer(
        address indexed from, 
        address indexed to, 
        uint256 indexed tokenId
    );
    event Approval(
        address indexed owner, 
        address indexed approved, 
        uint256 indexed tokenId
    );
    event ApprovalForAll(
        address indexed owner, 
        address indexed operator, 
        bool approved
    );

    function balanceOf(address owner) external view returns (uint256);
    function ownerOf(uint256 tokenId) external view returns (address);
    function safeTransferFrom(address from, address to, uint256 tokenId) external;
    function transferFrom(address from, address to, uint256 tokenId) external;
    function approve(address to, uint256 tokenId) external;
    function getApproved(uint256 tokenId) external view returns (address);
    function setApprovalForAll(address operator, bool approved) external;
    function isApprovedForAll(address owner, address operator) external view returns (bool);
}
'''

print("Interface ERC-721:")
print(ERC721_INTERFACE)

In [ ]:
# Implementation simplifiee ERC-721
ERC721_IMPLEMENTATION = '''
contract SimpleNFT is IERC721 {
    string public name = "Simple NFT";
    string public symbol = "SNFT";

    // Mapping tokenId => owner
    mapping(uint256 => address) private _owners;
    // Mapping owner => count
    mapping(address => uint256) private _balances;
    // Mapping tokenId => approved address
    mapping(uint256 => address) private _tokenApprovals;
    // Mapping owner => operator => approved
    mapping(address => mapping(address => bool)) private _operatorApprovals;

    uint256 private _nextTokenId = 1;

    function balanceOf(address owner) public view override returns (uint256) {
        require(owner != address(0), "ERC721: zero address");
        return _balances[owner];
    }

    function ownerOf(uint256 tokenId) public view override returns (address) {
        address owner = _owners[tokenId];
        require(owner != address(0), "ERC721: nonexistent token");
        return owner;
    }

    function approve(address to, uint256 tokenId) public override {
        address owner = ownerOf(tokenId);
        require(to != owner, "ERC721: approval to current owner");
        require(
            msg.sender == owner || isApprovedForAll(owner, msg.sender),
            "ERC721: not approved"
        );
        _tokenApprovals[tokenId] = to;
        emit Approval(owner, to, tokenId);
    }

    function getApproved(uint256 tokenId) public view override returns (address) {
        require(_owners[tokenId] != address(0), "ERC721: nonexistent token");
        return _tokenApprovals[tokenId];
    }

    function setApprovalForAll(address operator, bool approved) public override {
        _operatorApprovals[msg.sender][operator] = approved;
        emit ApprovalForAll(msg.sender, operator, approved);
    }

    function isApprovedForAll(address owner, address operator) 
        public view override returns (bool) 
    {
        return _operatorApprovals[owner][operator];
    }

    function transferFrom(address from, address to, uint256 tokenId) public override {
        require(_isApprovedOrOwner(msg.sender, tokenId), "ERC721: not approved");
        _transfer(from, to, tokenId);
    }

    function safeTransferFrom(address from, address to, uint256 tokenId) 
        public override 
    {
        transferFrom(from, to, tokenId);
        require(
            to.code.length == 0 || 
            IERC721Receiver(to).onERC721Received(msg.sender, from, tokenId, "") 
                == IERC721Receiver.onERC721Received.selector,
            "ERC721: transfer to non ERC721Receiver"
        );
    }

    // Mint public
    function mint() public returns (uint256) {
        uint256 tokenId = _nextTokenId++;
        _mint(msg.sender, tokenId);
        return tokenId;
    }

    // Internal functions
    function _mint(address to, uint256 tokenId) internal {
        require(to != address(0), "ERC721: mint to zero");
        require(_owners[tokenId] == address(0), "ERC721: token exists");
        
        _balances[to] += 1;
        _owners[tokenId] = to;
        emit Transfer(address(0), to, tokenId);
    }

    function _transfer(address from, address to, uint256 tokenId) internal {
        require(ownerOf(tokenId) == from, "ERC721: wrong owner");
        require(to != address(0), "ERC721: transfer to zero");
        
        _tokenApprovals[tokenId] = address(0);
        _balances[from] -= 1;
        _balances[to] += 1;
        _owners[tokenId] = to;
        
        emit Transfer(from, to, tokenId);
    }

    function _isApprovedOrOwner(address spender, uint256 tokenId) 
        internal view returns (bool) 
    {
        address owner = ownerOf(tokenId);
        return (
            spender == owner ||
            getApproved(tokenId) == spender ||
            isApprovedForAll(owner, spender)
        );
    }
}

// Interface pour les contrats recevant NFT
interface IERC721Receiver {
    function onERC721Received(
        address operator,
        address from,
        uint256 tokenId,
        bytes calldata data
    ) external returns (bytes4);
}
'''

print("Implementation ERC-721 simplifiee:")
print(ERC721_IMPLEMENTATION)

---

## 3. Standard ERC-1155 (Multi-Token)

ERC-1155 permet de gerer plusieurs types de tokens dans un seul contrat.

In [ ]:
# Apercu ERC-1155
ERC1155_OVERVIEW = '''
interface IERC1155 {
    event TransferSingle(
        address indexed operator,
        address indexed from,
        address indexed to,
        uint256 id,
        uint256 value
    );
    event TransferBatch(
        address indexed operator,
        address indexed from,
        address indexed to,
        uint256[] ids,
        uint256[] values
    );

    function balanceOf(address account, uint256 id) 
        external view returns (uint256);
    
    function balanceOfBatch(
        address[] calldata accounts,
        uint256[] calldata ids
    ) external view returns (uint256[] memory);
    
    function safeTransferFrom(
        address from,
        address to,
        uint256 id,
        uint256 amount,
        bytes calldata data
    ) external;
    
    function safeBatchTransferFrom(
        address from,
        address to,
        uint256[] calldata ids,
        uint256[] calldata amounts,
        bytes calldata data
    ) external;
}

// Avantages ERC-1155:
// - Un seul contrat pour FT et NFT
// - Transferts batch (gas efficient)
// - Idem pour mint/burn batch
// - Utilise pour les jeux (items, ressources)
'''

print("Apercu ERC-1155 (Multi-Token):")
print(ERC1155_OVERVIEW)

---

## 4. OpenZeppelin

OpenZeppelin fournit des implementations securisees et auditees.

In [ ]:
# Utilisation d'OpenZeppelin
OPENZEPPELIN_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

// Import depuis @openzeppelin/contracts
import "@openzeppelin/contracts/token/ERC20/ERC20.sol";
import "@openzeppelin/contracts/token/ERC20/extensions/ERC20Burnable.sol";
import "@openzeppelin/contracts/access/Ownable.sol";

// Token ERC-20 avec fonctionnalites etendues
contract MyToken is ERC20, ERC20Burnable, Ownable {
    uint8 private constant _decimals = 6;
    uint256 private constant INITIAL_SUPPLY = 1_000_000 * 10 ** _decimals;

    constructor() ERC20("My Token", "MTK") Ownable(msg.sender) {
        _mint(msg.sender, INITIAL_SUPPLY);
    }

    // Mint reserve au owner
    function mint(address to, uint256 amount) public onlyOwner {
        _mint(to, amount);
    }

    function decimals() public pure override returns (uint8) {
        return _decimals;
    }
}
'''

OPENZEPPELIN_NFT = '''
import "@openzeppelin/contracts/token/ERC721/ERC721.sol";
import "@openzeppelin/contracts/token/ERC721/extensions/ERC721URIStorage.sol";
import "@openzeppelin/contracts/access/Ownable.sol";

contract MyNFT is ERC721, ERC721URIStorage, Ownable {
    uint256 private _nextTokenId;

    constructor() 
        ERC721("My NFT", "MNFT") 
        Ownable(msg.sender) 
    {}

    function mint(string memory uri) public returns (uint256) {
        uint256 tokenId = _nextTokenId++;
        _safeMint(msg.sender, tokenId);
        _setTokenURI(tokenId, uri);
        return tokenId;
    }

    // Overrides requis
    function tokenURI(uint256 tokenId) 
        public view override(ERC721, ERC721URIStorage) 
        returns (string memory) 
    {
        return super.tokenURI(tokenId);
    }

    function supportsInterface(bytes4 interfaceId)
        public view override(ERC721, ERC721URIStorage)
        returns (bool)
    {
        return super.supportsInterface(interfaceId);
    }
}
'''

print("Token ERC-20 avec OpenZeppelin:")
print(OPENZEPPELIN_EXAMPLE)
print("\n" + "="*50 + "\n")
print("NFT ERC-721 avec OpenZeppelin:")
print(OPENZEPPELIN_NFT)

---

## 5. Exercices

### Exercice 1 : Token avec plafond

Creez un ERC-20 avec un supply maximum (capped).

In [ ]:
# Solution Exercice 1
SOLUTION_CAPPED = '''
contract CappedToken is IERC20 {
    string public constant name = "Capped Token";
    string public constant symbol = "CAP";
    uint8 public constant decimals = 18;
    
    uint256 public immutable cap;        // Maximum supply
    uint256 private _totalSupply;
    mapping(address => uint256) private _balances;
    mapping(address => mapping(address => uint256)) private _allowances;

    constructor(uint256 _cap) {
        require(_cap > 0, "Cap must be positive");
        cap = _cap;
    }

    function mint(address to, uint256 amount) public {
        require(_totalSupply + amount <= cap, "Cap exceeded");
        _totalSupply += amount;
        _balances[to] += amount;
        emit Transfer(address(0), to, amount);
    }

    // ... reste de l'implementation ERC-20 identique
}
'''
print("Solution Capped Token:")
print(SOLUTION_CAPPED)

### Exercice 2 : NFT avec enumeration

Creez un NFT qui permet d'enumerer tous les tokens.

In [ ]:
# Solution Exercice 2
SOLUTION_ENUMERABLE = '''
contract EnumerableNFT {
    mapping(uint256 => address) private _owners;
    mapping(address => uint256[]) private _ownedTokens;
    mapping(uint256 => uint256) private _ownedTokensIndex;
    uint256[] private _allTokens;
    mapping(uint256 => uint256) private _allTokensIndex;

    function totalSupply() public view returns (uint256) {
        return _allTokens.length;
    }

    function tokenByIndex(uint256 index) public view returns (uint256) {
        require(index < _allTokens.length, "Index out of bounds");
        return _allTokens[index];
    }

    function tokenOfOwnerByIndex(address owner, uint256 index) 
        public view returns (uint256) 
    {
        require(index < _ownedTokens[owner].length, "Index out of bounds");
        return _ownedTokens[owner][index];
    }

    function _mint(address to, uint256 tokenId) internal {
        _owners[tokenId] = to;
        
        // Add to owner's list
        _ownedTokens[to].push(tokenId);
        _ownedTokensIndex[tokenId] = _ownedTokens[to].length - 1;
        
        // Add to global list
        _allTokens.push(tokenId);
        _allTokensIndex[tokenId] = _allTokens.length - 1;
    }
}
'''
print("Solution Enumerable NFT:")
print(SOLUTION_ENUMERABLE)

---

## 6. Resume

| Standard | Type | Cas d'usage |
|----------|------|-------------|
| ERC-20 | Fongible | Cryptomonnaies, governance tokens |
| ERC-721 | Non-fongible | NFTs, assets uniques |
| ERC-1155 | Multi | Jeux, items, FT + NFT |

### Fonctions cles ERC-20
- `transfer()` : Envoyer des tokens
- `approve()` : Autoriser un spender
- `transferFrom()` : Transfer via allowance

### Fonctions cles ERC-721
- `ownerOf()` : Proprietaire d'un NFT
- `safeTransferFrom()` : Transfer securise
- `approve()` : Autoriser un transfert

---

**Notebook suivant** : [SC-6-DeFi-Primitives](SC-6-DeFi-Primitives.ipynb)